# Perbandingan Empat Dataset pada Setiap Model

Notebook ini membaca skor mean outer 5-fold CV untuk seluruh dataset dari notebook komparasi PIDD, FHGDD, BRFSS, dan Sylhet. Std menunjukkan variasi antar-fold, bukan confidence interval. Skor antar-dataset tetap perlu dibaca dengan memperhatikan fitur, populasi, dan prevalensi kelas positif.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid')
DATASET_ORDER = ['PIDD', 'FHGDD', 'BRFSS', 'Sylhet']
MODEL_ORDER = ['AdaBoost', 'XGBoost', 'LightGBM', 'CatBoost']
from cross_validation_utils import CV_PROTOCOL


## 1. Gabungkan hasil dan profil dataset

Metrik dibaca langsung dari file comparison_*_results.csv terbaru yang dibuat oleh notebook perbandingan masing-masing dataset. Profil data mengikuti preprocessing sumber: FHGDD dideduplikasi, sedangkan PIDD dan BRFSS tidak dideduplikasi. Exact duplicate patterns pada BRFSS tidak otomatis berarti responden yang sama, sehingga tidak dihapus.

In [ ]:
dataset_config = {
    'PIDD': {
        'data_file': 'PIDD_diabetes.csv',
        'result_file': 'comparison_PIDD_results.csv',
        'prediction_file': 'comparison_PIDD_predictions.csv',
        'target': 'Outcome', 'positive_value': 1,
        'drop_duplicates': False,
        'duplicate_handling': 'Tidak dihapus sesuai notebook sumber'
    },
    'FHGDD': {
        'data_file': 'frankfurt_diabetes.csv',
        'result_file': 'comparison_FHGDD_results.csv',
        'prediction_file': 'comparison_FHGDD_predictions.csv',
        'target': 'Outcome', 'positive_value': 1,
        'drop_duplicates': True,
        'duplicate_handling': 'Exact duplicate dihapus sebelum split'
    },
    'BRFSS': {
        'data_file': 'diabetes_binary_health_indicators_BRFSS2015.csv',
        'result_file': 'comparison_BRFSS_results.csv',
        'prediction_file': 'comparison_BRFSS_predictions.csv',
        'target': 'Diabetes_binary', 'positive_value': 1,
        'drop_duplicates': False,
        'duplicate_handling': 'Tidak dihapus sesuai notebook sumber'
    },
    'Sylhet': {
        'data_file': 'sylhet_diabetes.csv',
        'result_file': 'comparison_Sylhet_results.csv',
        'prediction_file': 'comparison_Sylhet_predictions.csv',
        'target': 'class', 'positive_value': 'Positive',
        'drop_duplicates': False,
        'duplicate_handling': 'Dipertahankan dengan group-aware split'
    }
}

result_frames = []
profile_rows = []
for dataset in DATASET_ORDER:
    config = dataset_config[dataset]
    raw_data = pd.read_csv(config['data_file'])
    prepared_data = (
        raw_data.drop_duplicates().reset_index(drop=True)
        if config['drop_duplicates'] else raw_data.copy()
    )
    results = pd.read_csv(config['result_file'])
    if ('Evaluation Protocol' not in results or
            not results['Evaluation Protocol'].eq(CV_PROTOCOL).all()):
        raise ValueError(f'Jalankan ulang comparison_{dataset}.ipynb; CSV masih memakai evaluasi lama.')
    expected_folds = 5
    if (not results['Outer Folds'].eq(expected_folds).all() or
            not results['Inner Folds'].eq(expected_folds).all()):
        raise ValueError(f'Jumlah fold {dataset} tidak sesuai. Jalankan ulang model dan komparasi.')
    assert set(results['Model']) == set(MODEL_ORDER)
    results['Model'] = pd.Categorical(
        results['Model'], categories=MODEL_ORDER, ordered=True
    )
    results = results.sort_values('Model').reset_index(drop=True)
    results.insert(0, 'Dataset', dataset)
    result_frames.append(results)

    prediction_rows = pd.read_csv(config['prediction_file'])
    if len(prediction_rows) != len(prepared_data) or 'fold' not in prediction_rows:
        raise ValueError(f'Prediksi OOF {dataset} belum lengkap. Jalankan ulang notebook komparasi.')
    test_rows = len(prediction_rows)
    duplicate_patterns = int(raw_data.duplicated().sum())
    handling = config['duplicate_handling']
    profile_rows.append({
        'Dataset': dataset,
        'Raw Rows': len(raw_data),
        'Rows Used': len(prepared_data),
        'Features': prepared_data.shape[1] - 1,
        'Exact Duplicate Patterns': duplicate_patterns,
        'Duplicate Handling': handling,
        'Positive Prevalence': prepared_data[config['target']].eq(
            config['positive_value']
        ).mean(),
        'OOF Rows': test_rows
    })

all_results = pd.concat(result_frames, ignore_index=True)
all_results['Dataset'] = pd.Categorical(
    all_results['Dataset'], categories=DATASET_ORDER, ordered=True
)
all_results['Model'] = pd.Categorical(
    all_results['Model'], categories=MODEL_ORDER, ordered=True
)
all_results = all_results.sort_values(
    ['Dataset', 'Model']
).reset_index(drop=True)
profile_df = pd.DataFrame(profile_rows)
prevalence_map = profile_df.set_index('Dataset')['Positive Prevalence']

all_results['Positive Prevalence'] = (
    all_results['Dataset'].map(prevalence_map).astype(float)
)
all_results['PR-AUC Lift vs Prevalence'] = (
    all_results['PR-AUC'] / all_results['Positive Prevalence']
)
all_results.to_csv('comparison_datasets_per_model_results.csv', index=False)

assert len(all_results) == 16
assert all_results.groupby('Dataset')['Model'].nunique().eq(4).all()
profile_df

## 2. Tabel perbandingan untuk setiap model

In [ ]:
comparison_columns = [
    'Dataset', 'Model', 'CV F1', 'Accuracy', 'Balanced Accuracy',
    'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC',
    'PR-AUC Lift vs Prevalence', 'Accuracy Std', 'F1 Std',
]
all_results[comparison_columns].sort_values(['Model', 'Dataset']).round(4)

In [ ]:
per_model_columns = [
    'Dataset', 'Accuracy', 'Precision', 'Recall', 'F1',
    'ROC-AUC', 'PR-AUC', 'PR-AUC Lift vs Prevalence',
    'Accuracy Std', 'F1 Std'
]
for model in MODEL_ORDER:
    display(Markdown(f'### {model}'))
    model_table = (
        all_results.loc[all_results['Model'].eq(model), per_model_columns]
        .set_index('Dataset')
        .reindex(DATASET_ORDER)
    )
    display(model_table.round(4))

## 3. Diagram batang dataset pada setiap model

Grafik utama berikut menunjukkan Accuracy, Precision, Recall, dan F1 untuk keempat dataset pada masing-masing model.

In [ ]:
classification_metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
chart_font_settings = {
    'font.family': 'Times New Roman',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'legend.title_fontsize': 8,
    'figure.titlesize': 8
}
previous_chart_font = {
    key: plt.rcParams[key] for key in chart_font_settings
}
plt.rcParams.update(chart_font_settings)
fig, axes = plt.subplots(2, 2, figsize=(15, 11), sharey=True)
for ax, model in zip(axes.ravel(), MODEL_ORDER):
    model_plot = all_results.loc[
        all_results['Model'].eq(model), ['Dataset'] + classification_metrics
    ].melt(id_vars='Dataset', var_name='Metric', value_name='Score')
    sns.barplot(
        data=model_plot, x='Metric', y='Score', hue='Dataset',
        hue_order=DATASET_ORDER, ax=ax
    )
    ax.set_ylim(0, 1)
    ax.set_title(model)
    ax.set_xlabel('Metric')
    ax.set_ylabel('Score')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=8, padding=1)
    ax.legend(title='Dataset', loc='lower right')
fig.suptitle(
    'Dataset Comparison per Model',
    fontsize=8, y=1.01
)
plt.tight_layout()
plt.show()
plt.rcParams.update(previous_chart_font)

In [ ]:
heatmap_metrics = ['F1', 'ROC-AUC', 'PR-AUC', 'PR-AUC Lift vs Prevalence']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, metric in zip(axes.ravel(), heatmap_metrics):
    matrix = (
        all_results.pivot(index='Dataset', columns='Model', values=metric)
        .reindex(index=DATASET_ORDER, columns=MODEL_ORDER)
    )
    sns.heatmap(matrix, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax)
    ax.set_title(metric)
    ax.set_xlabel('Model')
    ax.set_ylabel('Dataset')
plt.tight_layout()
plt.show()

## 4. Ringkasan performa dan kestabilan dataset

Rata-rata merangkum empat model. Std fold yang lebih kecil berarti variasi skor antar-fold lebih kecil, bukan bukti estimasi independen. `PR-AUC Lift` membandingkan mean PR-AUC terhadap prevalensi positif.


In [ ]:
dataset_summary = (
    all_results.groupby('Dataset', as_index=False)
    .agg(
        Mean_Accuracy=('Accuracy', 'mean'),
        Mean_Balanced_Accuracy=('Balanced Accuracy', 'mean'),
        Mean_F1=('F1', 'mean'),
        Mean_ROC_AUC=('ROC-AUC', 'mean'),
        Mean_PR_AUC=('PR-AUC', 'mean'),
        Mean_PR_AUC_Lift=('PR-AUC Lift vs Prevalence', 'mean'),
        Mean_Accuracy_Std=('Accuracy Std', 'mean'),
        Mean_F1_Std=('F1 Std', 'mean'),
    )
    .merge(
        profile_df[['Dataset', 'Rows Used', 'Features', 'Positive Prevalence', 'OOF Rows']],
        on='Dataset', how='left'
    )
)
dataset_summary['Dataset'] = pd.Categorical(
    dataset_summary['Dataset'], categories=DATASET_ORDER, ordered=True
)
dataset_summary = dataset_summary.sort_values('Dataset').reset_index(drop=True)
dataset_summary.to_csv('comparison_dataset_summary.csv', index=False)
dataset_summary.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.barplot(data=profile_df, x='Dataset', y='Rows Used', order=DATASET_ORDER, ax=axes[0])
axes[0].set_yscale('log')
axes[0].set_title('Number of Rows Used (Log Scale)')
axes[0].set_xlabel('Dataset')
axes[0].set_ylabel('Number of Rows')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.0f', padding=2)

sns.barplot(
    data=profile_df, x='Dataset', y='Positive Prevalence',
    order=DATASET_ORDER, ax=axes[1]
)
axes[1].set_ylim(0, 0.7)
axes[1].set_title('Diabetes Prevalence')
axes[1].set_xlabel('Dataset')
axes[1].set_ylabel('Positive Class Prevalence')
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.3f', padding=2)

std_plot = dataset_summary.melt(
    id_vars='Dataset',
    value_vars=['Mean_Accuracy_Std', 'Mean_F1_Std'],
    var_name='Metric', value_name='Mean Fold Std'
)
std_plot['Metric'] = std_plot['Metric'].map({
    'Mean_Accuracy_Std': 'Accuracy Fold Std',
    'Mean_F1_Std': 'F1 Fold Std'
})
sns.barplot(
    data=std_plot, x='Dataset', y='Mean Fold Std', hue='Metric',
    order=DATASET_ORDER, ax=axes[2]
)
axes[2].set_title('Mean Standard Deviation Across Folds')
axes[2].set_xlabel('Dataset')
axes[2].set_ylabel('Mean Fold Std')
axes[2].legend(title='')
plt.tight_layout()
plt.show()

## 5. Dataset terbaik menurut tujuan

Tidak digunakan McNemar antar-dataset karena observasi test-nya berbeda dan tidak berpasangan. Pemenang berikut harus dibaca sebagai panduan bersyarat, bukan bukti bahwa satu dataset universal lebih baik.

In [ ]:
decision_rows = []
for model in MODEL_ORDER:
    model_results = all_results.loc[all_results['Model'].eq(model)]
    decision_rows.append({
        'Model': model,
        'Raw F1 tertinggi': model_results.loc[model_results['F1'].idxmax(), 'Dataset'],
        'ROC-AUC tertinggi': model_results.loc[model_results['ROC-AUC'].idxmax(), 'Dataset'],
        'PR-AUC tertinggi': model_results.loc[model_results['PR-AUC'].idxmax(), 'Dataset'],
        'PR-AUC lift tertinggi': model_results.loc[
            model_results['PR-AUC Lift vs Prevalence'].idxmax(), 'Dataset'
        ],
        'F1 std terkecil': model_results.loc[model_results['F1 Std'].idxmin(), 'Dataset']
    })
decision_df = pd.DataFrame(decision_rows)
display(decision_df)

raw_performance_winner = dataset_summary.loc[dataset_summary['Mean_F1'].idxmax(), 'Dataset']
roc_winner = dataset_summary.loc[dataset_summary['Mean_ROC_AUC'].idxmax(), 'Dataset']
stability_winner = dataset_summary.loc[dataset_summary['Mean_F1_Std'].idxmin(), 'Dataset']
relative_pr_winner = dataset_summary.loc[dataset_summary['Mean_PR_AUC_Lift'].idxmax(), 'Dataset']
largest_dataset = dataset_summary.loc[dataset_summary['Rows Used'].idxmax(), 'Dataset']

print(f'Skor F1 mentah rata-rata tertinggi : {raw_performance_winner}')
print(f'ROC-AUC rata-rata tertinggi        : {roc_winner}')
print(f'F1 std antar-fold terkecil  : {stability_winner}')
print(f'PR-AUC lift rata-rata tertinggi   : {relative_pr_winner}')
print(f'Dataset terbesar                  : {largest_dataset}')
print()
print('Interpretasi:')
print('- Skor mentah tidak langsung sebanding karena fitur, populasi, dan prevalensi berbeda.')
print('- Sylhet memakai group-aware split agar profil gejala identik tidak bocor antarsplit.')
print('- Pilihan akhir harus mengikuti populasi target dan fitur yang tersedia saat deployment.')

### Cara mengambil keputusan

- Gunakan **FHGDD** jika fokusnya adalah eksperimen pada distribusi FHGDD dan Anda menerima ketidakpastian akibat sampel kecil.
- Gunakan **BRFSS** jika fokusnya adalah fondasi pelatihan yang besar, estimasi yang stabil, dan fitur survei BRFSS memang tersedia pada penggunaan nyata.
- Gunakan **PIDD** bila populasi serta delapan fitur klinis PIDD paling sesuai dengan kasus penggunaan.
- Untuk klaim generalisasi, lakukan validasi eksternal: latih pada satu dataset dan uji pada dataset lain yang memiliki definisi fitur/target kompatibel.